## First test of VA in chamber

In [ ]:
# Imports
import numpy as np
import time

from mcg_lab_vna_control import connect_vna
from mcg_lab_triaxial_positioner_control import TriaxialPositioner

# Transparent conection: the library knows witch driver to use
vna = connect_vna(brand='keysight')

In [ ]:
########## Parameters ##########
# Triaxial Positioner Port
PORT = "COM6"

# VNA Parameters
ifbw = 1e3
pow_dbm = 0
f_start_list = [] # Segmented f_start lis (in Hz)
f_stop_list = [] # Segmented f_stop lis (in Hz)
points_list = [] # Segmented points list

# Folder name for saving measurement files
folder_name = 'VA_test'

### Basic VNA Config to see Time-Response of the Channel

In [ ]:
vna.preset()

vna.configure_sparam(sparam_str = 'S21', trace = 1, f_start = 10e9, f_stop = 15e9, points = 4001, ifbw = 10e3, pow_dbm = pow_dbm)
vna.configure_sparam(sparam_str = 'S22', trace = 2, f_start = 10e9, f_stop = 15e9, points = 4001, ifbw = 10e3, pow_dbm = pow_dbm)

vna.run_cont()

vna.set_timedomain(trace = 1)
vna.set_timedomain(trace = 2)

vna.auto_scale_all()

In [ ]:
vna.set_Xscale(xmin = 150, xmax = 190, mode = 'dist', trace = 1)
vna.set_Xscale(xmin = 0, xmax = 20, mode = 'dist', trace = 1)
vna.auto_scale_all()

### Calibrate the VNA for Segmented Measurements

In [ ]:
vna.preset()

vna.configure_segmented_sparam(sparam_str = 'S21', trace = 1, 
                               f_start_list = f_start_list, f_stop_list = f_stop_list, 
                               points_list = points_list, ifbw = ifbw, pow_dbm = pow_dbm)

vna.configure_segmented_sparam(sparam_str = 'S22', trace = 2, 
                               f_start_list = f_start_list, f_stop_list = f_stop_list,
                               points_list = points_list, ifbw = ifbw, pow_dbm = pow_dbm)

vna.init_SOLT_calibration(conn_type = '2.92', conn_gender = 'ffnn', calkit = 'ZV-Z129')

In [ ]:
# Do it with all calibrators numbers
vna.measure_SOLT_calibrator(7)

In [ ]:
# Apply calibration
vna.apply_calibration()

### Measure S21 of Through for post-calibration

In [ ]:
# Measureme S21 of Through
sparam = vna.measure_trace_sparam(trace=1)
vna.auto_scale_all()

vna.save_s1p(filename= 'through.s1p', folder = folder_name, freqs = vna.get_freq_vector(), sdata = sparam, sparam_str = 'S21')

### VA Grid Measurement Loop

In [ ]:
########## Parameters ##########
setup_number = 1 # setup number (to have different setup trials)
va_vert_pos = 0 # 0 | 1, idx position in vertical of antenna of the VA  
va_deg = 0 # degrees of boresight of VA antenna
ue_deg = 0

va_Nh = 30 # VA horizontal (Nh) positions
va_Nv = 15 # VA vertical (Nv) positions
va_deltah = 10.0 # mm of Delta H
va_deltav = 10.0 # mm of Delta V

In [ ]:
# The 'with' statement automatically handles connection and safe disconnection
with TriaxialPositioner(port=PORT) as pos:
    print("Connected to C4 Controller.")
    
    # Configure velocity (optional)
    pos.set_velocity(axis='X', min_speed=200, max_speed=1500, slope=2000)

    # Move to the limit
    pos.move_axes_relative(x=-1000,y=-1000,z=-1000, unit='mm')

    # Establish Absolute Zero
    print("Defining current location as Origin (0,0,0)...")
    pos.define_global_coord(x=0.0, y=0.0, z=0.0, unit='mm')

    # Executing a 2D Grid Scan using Generators
    print("Starting 2D Grid Scan...")
    grid_iterator = pos.generate_grid_2d(
        axis1='X', 
        axis2='Z', 
        points1=va_Nh, 
        points2=va_Nv, 
        spacing1=va_deltah, 
        spacing2=va_deltav, 
        unit='mm'
    )
    
    for step in grid_iterator:
        print(f"Arrived at Point {step['point']}/{step['total']} | X: {step['X']}mm, Z: {step['Z']}mm")

        step_i = step['i']
        step_j = step['j'] + va_vert_pos * va_Nv

        # Current Step Filenames
        filename_s21 = f"setup_{setup_number}_S21_nh_{step_i}_nv_{step_j}_VA_{va_deg}_deg_UE_{ue_deg}deg"
        filename_s22 = f"setup_{setup_number}_S22_nh_{step_i}_nv_{step_j}_VA_{va_deg}_deg_UE_{ue_deg}deg"

        # Measure S21 (Bi BS-UE)
        sparam = vna.measure_trace_sparam(trace=1)
        vna.auto_scale_all()

        vna.save_s1p(filename = filename_s21, folder = folder_name, freqs = vna.get_freq_vector(), sdata = sparam, sparam_str = 'S21')

        time.sleep(1)

        # Measure S22 (Mono UE-UE)
        sparam = vna.measure_trace_sparam(trace=2)
        vna.auto_scale_all()

        vna.save_s1p(filename = filename_s22, folder = folder_name, freqs = vna.get_freq_vector(), sdata = sparam, sparam_str = 'S22')

        time.sleep(1) 
    
    print("Grid scan complete!")

    # Return Home
    print("Returning to origin...")
    pos.move_axes_absolute(x=0.0, y=0.0, z=0.0, wait_to_finish=True, unit='mm')

print("Routine finished and hardware disconnected safely.")